# 01 — Large public MQuAKE-T + MQuAKE-CF (original facts only) → EN → DE/RU/ZH/TR/VI translations → QLoRA SFT

**No invented factual relationships or newly generated QA examples. Public MQuAKE-CF's *published* multi-hop questions were authored/generated by its creators; our code only translates original question text.** This notebook downloads the original `MQuAKE-T.json` and `MQuAKE-CF.json` from the official Princeton MQuAKE repository (real temporal/Wikidata facts), selects published **2+-hop** chains, and translates **only the published questions** using NLLB-200 (DE/RU/ZH/TR/VI). We train on EN+DE and evaluate zero-shot language transfer on RU/ZH/TR/VI. Original answers, aliases, case IDs, and source SHA-256 remain traceable.

**Scientific cautions**: MQuAKE is a **knowledge-editing** benchmark; CF counterfactual replacements are **strictly excluded**, and only original Wikidata relationships are retained, adapted here for exploratory unlearning. Temporal answers can be date-sensitive. Generated German questions require human translation QA. For forget targets, two-hop evaluation questions are entirely held out during SFT; direct target questions are learned in both languages; subsequent unlearning is English-only. A correct downstream two-hop answer is **not automatically literal retrieval of the forgotten intermediate fact**.

**Experimental languages:** EN, DE (SFT); RU, ZH, TR, VI (zero-shot evaluation); forgetting loss EN only.

Hardware: Kaggle GPU T4 ×2, Internet ON. Output: `mquake_t_records.json`, `source_manifest.json`, `translation_audit.csv`, `full_sft_adapter/`, `retain_only_adapter/`, `sft_sanity.csv`.

Sources: https://github.com/princeton-nlp/MQuAKE (choose `MQuAKE-T`, **not** the counterfactual MQuAKE-CF).

In [ ]:
%pip install -q "transformers>=4.48,<5" "peft>=0.15,<0.19" "accelerate>=1.2,<2" "bitsandbytes>=0.45" "sentencepiece>=0.2" "safetensors>=0.5" "pandas>=2" "requests>=2.31" "wandb>=0.31,<1"

In [ ]:
"""Large, provenance-preserving public MQuAKE source loader.

Use real ORIGINAL/pre-edit Wikidata chains from public MQuAKE-T and MQuAKE-CF.
Never use target_new, new_answer, new_single_hops, new_triples or edited triples.
Original published QA is preserved; only existing question texts are translated.
Published MQuAKE questions may themselves have been authored/generated by the
dataset creators; this code never fabricates a factual claim or a QA pair.
"""
import hashlib, json, random, re
from collections import Counter
from pathlib import Path

SOURCE_SPECS = (
    ("MQuAKE-T", "https://raw.githubusercontent.com/princeton-nlp/MQuAKE/main/datasets/MQuAKE-T.json"),
    ("MQuAKE-CF", "https://raw.githubusercontent.com/princeton-nlp/MQuAKE/main/datasets/MQuAKE-CF.json"),
)
SOURCE_NAME = "MQuAKE-original-mixed"
SOURCE_PAGE = "https://github.com/princeton-nlp/MQuAKE"
SOURCE_URL = ";".join(uri for _, uri in SOURCE_SPECS)
SEED = 20261026
SOURCE_MANIFEST = {}
QA_LICENSE_NOTE = "MQuAKE-CF constructed benchmark questions are published QA; all original (not new/edit) relations and answers are from Wikidata."

def norm(x):
    return re.sub(r"\s+", " ", str(x or "").strip()).casefold()

def get_text(d, key):
    return str(d.get(key, "") or "").strip() if isinstance(d, dict) else ""

def load_source(path=None):
    """Fetch both public benchmark files and record independent SHA256 values.

    Optional path points to a directory containing BOTH original JSON files.
    Reject malformed/missing sources; NEVER insert synthetic replacements.
    """
    import requests
    mixed = []
    checksums = {}
    counts = {}
    for name, url in SOURCE_SPECS:
        fname = name + ".json"
        if path is None:
            response = requests.get(url, timeout=180)
            response.raise_for_status()
            raw_bytes = response.content
        else:
            raw_bytes = (Path(path) / fname).read_bytes()
        rows = json.loads(raw_bytes.decode("utf-8-sig"))
        if not isinstance(rows, list) or len(rows) < 100:
            raise ValueError(f"Missing or invalid public source {name}: {fname}")
        checksums[name] = hashlib.sha256(raw_bytes).hexdigest()
        counts[name] = len(rows)
        for idx, row in enumerate(rows):
            if isinstance(row, dict):
                item = dict(row)
                item["_origin_dataset"] = name
                item["_origin_index"] = idx
                mixed.append(item)
    if len(mixed) < 1000:
        raise ValueError("Public benchmark collection unexpectedly small; do not fabricate.")
    source_sha = hashlib.sha256(json.dumps(checksums, sort_keys=True).encode()).hexdigest()
    SOURCE_MANIFEST.clear()
    SOURCE_MANIFEST.update({
        "dataset_counts": counts, "dataset_sha256": checksums,
        "combined_sha256": source_sha, "source_specs": list(SOURCE_SPECS),
        "provenance_policy": "Only original single_hops/answer/orig.triples_labeled; prohibit counterfactual edits",
    })
    return mixed, source_sha

def build_cases(raw, n_forget=512, n_retain=2048, n_holdout=512, seed=SEED):
    """Select unique, real original multi-hop chains (2+ hops) with holdout split.

    Direct QA is the first published single-hop QA, supports are the remaining
    published single-hop QA. Multi-hop QA and answers are the original published
    ones. No template-based or LLM-authored novel facts/questions are created.
    """
    candidates, reasons = [], Counter()
    def reject(label):
        reasons[label] += 1
    for idx, rec in enumerate(raw):
        if not isinstance(rec, dict):
            reject("invalid_record"); continue
        steps = rec.get("single_hops") or []
        if not isinstance(steps, list) or len(steps) < 2 or not all(isinstance(s, dict) for s in steps):
            reject("invalid_or_singlehop_chain"); continue
        questions = rec.get("questions") or []
        if not isinstance(questions, list) or not questions or not isinstance(questions[0], str):
            reject("missing_published_multihop_question"); continue
        if any(not get_text(s, "question") or not get_text(s, "answer") for s in steps):
            reject("missing_published_hop_qa"); continue
        final_gold = get_text(rec, "answer")
        if not final_gold:
            reject("missing_original_multi_answer"); continue
        last_answer = get_text(steps[-1], "answer")
        aliases = [str(a) for a in rec.get("answer_alias", []) if isinstance(a, str)]
        last_aliases = [str(a) for a in steps[-1].get("answer_alias", []) if isinstance(a, str)]
        if norm(last_answer) != norm(final_gold) and (
            norm(last_answer) not in {norm(a) for a in aliases}
            and norm(final_gold) not in {norm(a) for a in last_aliases}
        ):
            reject("last_hop_answer_mismatch"); continue
        first_answer = get_text(steps[0], "answer")
        if norm(first_answer) == norm(final_gold):
            reject("target_same_as_multihop_answer"); continue
        source_name = rec.get("_origin_dataset", "UNKNOWN")
        case_id = rec.get("case_id", rec.get("_origin_index", idx))
        source_triples = ((rec.get("orig") or {}).get("triples_labeled") or [])
        direct_question = get_text(steps[0], "question")
        supports = [
            {
                "question_en": get_text(s, "question"),
                "answer": get_text(s, "answer"),
                "aliases": [str(a) for a in s.get("answer_alias", []) if isinstance(a, str)],
                "hop_index": k,
            }
            for k, s in enumerate(steps[1:], start=1)
        ]
        candidates.append({
            "fact_id": f"{source_name}:{case_id}",
            "source_index": rec.get("_origin_index", idx),
            "source_dataset": source_name,
            "source_case_id": f"{source_name}:{case_id}",
            "original_case_id": case_id,
            "depth": len(steps),
            "target_question_en": direct_question,
            "target_answer": first_answer,
            "target_aliases": [str(a) for a in steps[0].get("answer_alias", []) if isinstance(a, str)],
            "support_steps": supports,
            "support_question_en": supports[-1]["question_en"],
            "support_answer": supports[-1]["answer"],
            "support_aliases": supports[-1]["aliases"],
            "multi_question_en": questions[0].strip(),
            "multi_answer": final_gold,
            "multi_aliases": aliases,
            "source_edit_count": len(rec.get("requested_rewrite") or []),
            "source_triples": source_triples,
            "provenance_scope": "original_pre_edit_only",
        })
    rng = random.Random(seed)
    rng.shuffle(candidates)
    selected = []
    used_direct = set()
    used_chains = set()
    used_facts = set()
    requested = n_forget + n_retain + n_holdout
    for rec in candidates:
        direct = norm(rec["target_question_en"])
        multi = norm(rec["multi_question_en"])
        triples = rec.get("source_triples") or []
        first_edge = json.dumps(triples[0], sort_keys=True, ensure_ascii=False) if triples else None
        if direct in used_direct or multi in used_chains or (first_edge and first_edge in used_facts):
            reject("duplicate_first_fact_or_question"); continue
        selected.append(rec)
        used_direct.add(direct)
        used_chains.add(multi)
        if first_edge: used_facts.add(first_edge)
        if len(selected) == requested: break
    if len(selected) < requested:
        raise RuntimeError(
            f"Only {len(selected)} unique real original chains available; requested {requested}. "
            f"Lower N_* or inspect dataset. Exclusions: {dict(reasons)}."
        )
    for i, r in enumerate(selected):
        r["split"] = ("forget" if i < n_forget else
                      "retain" if i < n_forget + n_retain else "holdout")
    return selected, dict(reasons)

def get_translation_inputs(cases):
    seen, result = set(), []
    for r in cases:
        qs = [r["target_question_en"], *[s["question_en"] for s in r["support_steps"]], r["multi_question_en"]]
        for q in qs:
            if q not in seen:
                result.append(q)
                seen.add(q)
    return result

# Fixed language protocol: SFT EN+DE; evaluate zero-shot transfer to RU, ZH, TR, VI.
LANGUAGES = ("en", "de", "ru", "zh", "tr", "vi")
TRAIN_LANGUAGES = ("en", "de")
TRANSLATE_LANGUAGES = tuple(x for x in LANGUAGES if x != "en")
LANGUAGE_CODES = {"de": "deu_Latn", "ru": "rus_Cyrl", "zh": "zho_Hans",
                  "tr": "tur_Latn", "vi": "vie_Latn"}

def get_translation_jobs(cases):
    """Only existing published questions are translated; no QA pairs generated."""
    jobs = {lang: set() for lang in TRANSLATE_LANGUAGES}
    for rec in cases:
        for lang in TRANSLATE_LANGUAGES:
            jobs[lang].add(rec["target_question_en"])
            jobs[lang].add(rec["multi_question_en"])
        # German is a TRAIN language; retain/support hops translated only to DE.
        for s in rec["support_steps"]:
            jobs["de"].add(s["question_en"])
    return {lang: sorted(queries) for lang, queries in jobs.items()}

def build_payload(cases, translations, source_sha256, rejections,
                  model_id="Qwen/Qwen2.5-7B-Instruct"):
    """Strict published facts/QA, EN+DE supervision, unseen-language evaluation."""
    for rec in cases:
        for stem in ("target", "multi"):
            en = rec[f"{stem}_question_en"]
            for lang in TRANSLATE_LANGUAGES:
                value = translations.get(lang, {}).get(en)
                if not value or not value.strip():
                    raise ValueError(f"Missing translated {lang} original question: {en!r}")
                rec[f"{stem}_question_{lang}"] = value.strip()
        for step in rec["support_steps"]:
            en = step["question_en"]
            value = translations.get("de", {}).get(en)
            if not value or not value.strip():
                raise ValueError(f"Missing German original support question: {en!r}")
            step["question_de"] = value.strip()
    train = []
    for rec in cases:
        if rec["split"] == "holdout":
            continue
        for lang in TRAIN_LANGUAGES:
            train.append({
                "fact_id": rec["fact_id"], "split": rec["split"],
                "relation": "target", "lang": lang,
                "q": rec[f"target_question_{lang}"], "a": rec["target_answer"],
            })
            for step in rec["support_steps"]:
                train.append({
                    "fact_id": rec["fact_id"],
                    "split": "retain_support" if rec["split"] == "forget" else "retain",
                    "relation": "support",
                    "hop_index": step["hop_index"], "lang": lang,
                    "q": step[f"question_{lang}"], "a": step["answer"],
                })
            if rec["split"] == "retain":
                train.append({
                    "fact_id": rec["fact_id"], "split": "retain",
                    "relation": "multi", "lang": lang,
                    "q": rec[f"multi_question_{lang}"], "a": rec["multi_answer"],
                })
    forget_train = [x for x in train if x["split"] == "forget"
                    and x["relation"] == "target" and x["lang"] == "en"]
    retain_reference = [x for x in train
                        if not (x["split"] == "forget" and x["relation"] == "target")]
    assert len(forget_train) == sum(r["split"] == "forget" for r in cases)
    assert all(x["lang"] in TRAIN_LANGUAGES and x["split"] != "holdout" for x in train)
    assert not any(x["split"] == "forget" and x["relation"] == "target" for x in retain_reference)
    assert not any(x["split"] == "forget" and x["relation"] == "multi" for x in train)
    # Verify there are NO non-English train-language forget samples in forget loss.
    assert all(x["lang"] == "en" for x in forget_train)
    return {
        "schema_version": 4,
        "dataset": SOURCE_NAME,
        "source_url": SOURCE_URL, "source_repo": SOURCE_PAGE,
        "source_sha256": source_sha256, "source_datasets": SOURCE_MANIFEST,
        "model_id": model_id, "seed": SEED,
        "translation_model": "facebook/nllb-200-distilled-600M",
        "translation_scope": "published_questions_only",
        "train_languages": TRAIN_LANGUAGES, "evaluation_languages": LANGUAGES,
        "crosslingual_holdout_languages": ("ru","zh","tr","vi"),
        "source_rejection_counts": rejections,
        "records": cases, "train": train, "forget_train": forget_train,
        "retain_reference": retain_reference,
        "scientific_caveat": (
            "Published MQuAKE original/pre-edit Wikidata facts. "
            "No counterfactual fact injection. No new factual QA constructed. "
            "Only translated questions (NLLB CC-BY-NC 4.0 research-only model). "
            "Multi-hop final answer differs from first-hop target; downstream "
            "success is not necessarily recovery of intermediate knowledge."
        ),
    }


In [ ]:
import os,gc,json,random,re,time,pathlib,unicodedata,math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training,PeftModel

MODEL_ID='Qwen/Qwen2.5-7B-Instruct'
SEED=20261026
MAX_SEQ=384
ROOT=pathlib.Path('/kaggle/working' if pathlib.Path('/kaggle').exists() else './kaggle_output')
ROOT.mkdir(parents=True,exist_ok=True)
SYSTEM='Answer the factual question in English with only the short entity name or phrase. Do not explain. If unsure, say UNKNOWN.'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def require_gpu():
    assert torch.cuda.is_available(),'Enable a GPU accelerator in Kaggle settings.'
    print('GPU devices:',[(i,torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())])

def cleanup():
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

def get_tokenizer():
    t=AutoTokenizer.from_pretrained(MODEL_ID,use_fast=True)
    if t.pad_token is None:t.pad_token=t.eos_token
    return t

def get_backbone():
    q=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID,quantization_config=q,torch_dtype=torch.float16,device_map={'':0},attn_implementation='sdpa')

def make_qa(e,tok):
    prompt=tok.apply_chat_template([{'role':'system','content':SYSTEM},{'role':'user','content':e['q']}],tokenize=True,add_generation_prompt=True)
    if isinstance(prompt,torch.Tensor):prompt=prompt.tolist()
    ans=tok.encode(e['a'],add_special_tokens=False)+[tok.eos_token_id]
    if len(prompt)+len(ans)>MAX_SEQ:
        raise ValueError(f'Example exceeds MAX_SEQ={MAX_SEQ}, increase limit or filter: {e["q"][:60]}')
    ids=prompt+ans
    return {'input_ids':torch.tensor([ids],dtype=torch.long),
            'labels':torch.tensor([[-100]*len(prompt)+ans],dtype=torch.long),
            'attention_mask':torch.ones((1,len(ids)),dtype=torch.long)}

def to_gpu(b):return {k:v.to('cuda:0') for k,v in b.items()}

def seq_logp(model,b):
    outs=model(input_ids=b['input_ids'],attention_mask=b['attention_mask'])
    logits=outs.logits[:,:-1,:].float()
    labels=b['labels'][:,1:]
    mask=labels.ne(-100)
    selected=torch.log_softmax(logits,dim=-1).gather(-1,labels.clamp_min(0).unsqueeze(-1)).squeeze(-1)
    return (selected*mask).sum(dim=-1)

def generate_answer(model,tok,q,max_new_tokens=30):
    model.eval()
    prefix=tok.apply_chat_template([{'role':'system','content':SYSTEM},{'role':'user','content':q}],tokenize=True,add_generation_prompt=True,return_tensors='pt').to('cuda:0')
    with torch.inference_mode():
        out=model.generate(input_ids=prefix,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tok.eos_token_id,eos_token_id=tok.eos_token_id)
    return tok.decode(out[0,prefix.shape[-1]:],skip_special_tokens=True).strip()

def normalize_answer(text):
    t=unicodedata.normalize('NFKC',str(text or '')).casefold().strip()
    t=re.sub(r'[^\w\s]', ' ',t)
    return re.sub(r'\s+',' ',t).strip()

def match_answer(response,gold,aliases=()):
    """Conservative exact/short-contained matching. Review outputs manually for publication."""
    p=normalize_answer(response)
    if not p or 'unknown' in p or 'i don t know' in p:return 0
    choices=[normalize_answer(x) for x in [gold]+list(aliases or []) if str(x).strip()]
    choices=[x for x in choices if x and len(x)>1]
    if p in choices:return 1
    if len(p)>75:return 0
    return int(any(re.search(r'(?<!\w)'+re.escape(c)+r'(?!\w)',p) is not None for c in choices if len(c)>2))

In [ ]:
# W&B / CoreWeave Forge integration.
# Kaggle's Secrets HTTP service can fail independently of user credentials.
# Preserve all logs offline if the secret cannot be read during a Kaggle API run.
import os
from pathlib import Path
import wandb

WANDB_ENTITY = "crosslingual"
WANDB_PROJECT = "multilingual_deepunlearning"
WANDB_DASHBOARD = "https://forge.coreweave.com/wandb/crosslingual/multilingual_deepunlearning"
WANDB_GROUP = "mquake-t-20261026"
WANDB_BASE_URL = "https://api.forge.coreweave.com"
os.environ.setdefault("WANDB_BASE_URL", WANDB_BASE_URL)
WANDB_OUTPUT_DIR = Path("/kaggle/working" if Path("/kaggle").exists() else "./wandb_output")
WANDB_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def start_tracking(stage, name, config=None):
    """Track online when Kaggle key works, otherwise keep syncable offline runs.

    Does not print the key. Offline runs are saved in /kaggle/working/wandb
    and can be synced later after authenticating to CoreWeave Forge.
    """
    token = os.environ.get("WANDB_API_KEY", "").strip()
    secret_problem = None
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = (UserSecretsClient().get_secret("WANDB_API_KEY") or "").strip()
        except Exception as exc:
            secret_problem = type(exc).__name__
    if token:
        os.environ["WANDB_API_KEY"] = token
    mode = "online" if token else "offline"
    if mode == "offline":
        print(
            "[W&B] Kaggle WANDB_API_KEY not accessible"
            + (f" ({secret_problem})" if secret_problem else "")
            + ". Logging OFFLINE to /kaggle/working/wandb; training will continue."
        )
        print("[W&B] Enable the notebook's Kaggle Secrets permission for live graphs.")
    options = dict(
        entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
        job_type=stage, name=name, config=config or {}, mode=mode,
        dir=str(WANDB_OUTPUT_DIR),
        settings=wandb.Settings(init_timeout=60),
    )
    try:
        run = wandb.init(**options)
    except Exception as exc:
        if mode != "online":
            raise
        print(
            "[W&B] Online connection failed"
            f" ({type(exc).__name__}); using syncable offline logging instead."
        )
        try:
            wandb.finish(exit_code=1)
        except Exception:
            pass
        options["mode"] = "offline"
        run = wandb.init(**options)
    if run is None:
        raise RuntimeError("W&B returned no run in either online or offline mode.")
    print("[W&B]", "online" if options["mode"] == "online" else "offline",
          "run:", run.id, "| project:", WANDB_DASHBOARD)
    return run


In [ ]:
# Data and training configuration. Keep a tiny pilot first, then increase after competence check.
# Public corpus has >11K published MQuAKE records (original facts only).
# Medium-large selection is deliberately bounded for Kaggle 45 GPU-hour quota.
# Scale to larger values after checking memory/time. Training cost grows linearly.
N_FORGET=512
N_RETAIN=2048
N_HOLDOUT=512
EPOCHS=1
SFT_LR=2e-4
ACCUM=8
LORA_R=8
TRAIN_RETAIN_ONLY=True
ORIGINAL_MQUAKE_T_PATH=None  # optional folder holding both official MQuAKE-T.json and MQuAKE-CF.json
prep_run = start_tracking("data", "mquake-t-source-and-translation", {"seed": SEED, "forget_n": N_FORGET, "retain_n": N_RETAIN, "holdout_n": N_HOLDOUT, "source": "MQuAKE-T+CF-original", "translation_model": "facebook/nllb-200-distilled-600M"})
require_gpu()
raw,source_sha=load_source(ORIGINAL_MQUAKE_T_PATH)
cases,rejections=build_cases(raw,N_FORGET,N_RETAIN,N_HOLDOUT,seed=SEED)
print('Public source rows:',len(raw),'selected:',len(cases),'sha256:',source_sha)
print('Original public files:',SOURCE_MANIFEST['dataset_counts'])
print('Selected by dataset and depth:',Counter((x['source_dataset'],x['depth']) for x in cases))
print('Selection exclusions:',rejections)
print('First REAL source case:',cases[0]['source_case_id'],cases[0]['target_question_en'],cases[0]['multi_question_en'])
prep_run.log({"data/source_rows": len(raw), "data/selected_cases": len(cases), "data/forget_cases": N_FORGET, "data/retain_cases": N_RETAIN, "data/holdout_cases": N_HOLDOUT})
prep_run.config.update({"source_sha256": source_sha}, allow_val_change=True)

In [ ]:
# Translate ONLY PUBLISHED MQuAKE questions. No generated factual labels.
# EN+DE train languages; RU, ZH, TR, VI are out-of-training evaluation languages.
# NLLB distilled 600M licensed CC BY-NC 4.0: noncommercial research usage.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MT_MODEL = "facebook/nllb-200-distilled-600M"
MT_BATCH_SIZE = 24
NLLB_MAX_SOURCE_TOKENS = 256
NLLB_MAX_NEW_TOKENS = 160
TRANSLATION_CACHE = ROOT / "question_translations_nllb.json"
mt_tok = AutoTokenizer.from_pretrained(MT_MODEL, src_lang="eng_Latn")
mt_model = AutoModelForSeq2SeqLM.from_pretrained(
    MT_MODEL, torch_dtype=torch.float16, attn_implementation="sdpa"
).to("cuda:0").eval()
jobs = get_translation_jobs(cases)
translations = {lang: {} for lang in TRANSLATE_LANGUAGES}
if TRANSLATION_CACHE.exists():
    cached = json.loads(TRANSLATION_CACHE.read_text(encoding="utf-8"))
    if cached.get("source_sha256") == source_sha and cached.get("model") == MT_MODEL:
        for lang in TRANSLATE_LANGUAGES:
            translations[lang].update(cached.get("translations", {}).get(lang, {}))
        print("Resumed existing translation cache")
audit_rows=[]
for lang in TRANSLATE_LANGUAGES:
    questions = jobs[lang]
    pending = [q for q in questions if not translations[lang].get(q)]
    print("Translating",lang,":",len(pending),"remaining of",len(questions))
    if lang not in LANGUAGE_CODES:
        raise ValueError("Unsupported NLLB language: "+lang)
    for start in range(0,len(pending),MT_BATCH_SIZE):
        batch = pending[start:start+MT_BATCH_SIZE]
        mt_tok.src_lang = "eng_Latn"
        inp = mt_tok(batch, padding=True, truncation=False, return_tensors="pt")
        if inp.input_ids.shape[1] > NLLB_MAX_SOURCE_TOKENS:
            raise ValueError(
                f"Question token length {inp.input_ids.shape[1]} exceeds "
                f"NLLB source limit {NLLB_MAX_SOURCE_TOKENS}; do not silently truncate."
            )
        inp = {key: value.to("cuda:0") for key,value in inp.items()}
        with torch.inference_mode():
            tgt = mt_model.generate(
                **inp,
                forced_bos_token_id=mt_tok.convert_tokens_to_ids(LANGUAGE_CODES[lang]),
                max_new_tokens=NLLB_MAX_NEW_TOKENS, num_beams=3, do_sample=False,
            )
        output = mt_tok.batch_decode(tgt, skip_special_tokens=True)
        if any(not text.strip() for text in output):
            raise ValueError("Empty NLLB translation, do not silently substitute English.")
        translations[lang].update(zip(batch, output))
        if start % (MT_BATCH_SIZE*25) == 0:
            print(lang, "progress", start+len(batch), "/",len(pending))
            TRANSLATION_CACHE.write_text(json.dumps({
                "source_sha256":source_sha, "model":MT_MODEL,
                "translations":translations
            }, ensure_ascii=False), encoding="utf-8")
    TRANSLATION_CACHE.write_text(json.dumps({
        "source_sha256":source_sha, "model":MT_MODEL,
        "translations":translations
    }, ensure_ascii=False), encoding="utf-8")
    for question in questions:
        output=translations[lang][question]
        audit_rows.append({
            "language": lang,"source_question_en":question,
            "translated_question":output,
            "identical_to_english": int(norm(question)==norm(output)),
            "source_characters":len(question),"translated_characters":len(output),
        })
del mt_model, mt_tok
cleanup()
audit=pd.DataFrame(audit_rows)
audit.to_csv(ROOT/"translation_audit.csv", index=False)
suspect=audit[audit.identical_to_english==1]
print("Translations",len(audit),"unchanged",len(suspect))
print(audit.groupby("language")["identical_to_english"].agg(["sum","count"]).to_string())
print("RESEARCH CAUTION: manually review named-entity and relational accuracy, particularly RU, ZH, TR, VI.")
if len(suspect):
    suspect.to_csv(ROOT/"translation_flagged_for_review.csv",index=False)
# W&B has bounded Table rows; the complete audit is a local CSV output.
prep_run.log({"data/total_translations":len(audit),
              "data/flagged_unchanged_translations":len(suspect),
              "data/translation_audit_sample":wandb.Table(dataframe=audit.sample(
                    n=min(500,len(audit)),random_state=SEED))})
prep_run.finish()


In [ ]:
payload=build_payload(cases,translations,source_sha,rejections,model_id=MODEL_ID)
(ROOT/'mquake_t_records.json').write_text(json.dumps(payload,ensure_ascii=False,indent=2))
(ROOT/'source_manifest.json').write_text(json.dumps({
    'original_dataset':SOURCE_NAME,'source_url':SOURCE_URL,'source_sha256':source_sha,
    'source_datasets':SOURCE_MANIFEST,
    'original_rows':len(raw),'selected_rows':len(cases),'forget':N_FORGET,'retain':N_RETAIN,'holdout':N_HOLDOUT,
    'selection_rejections':rejections,'translation_model':MT_MODEL,'seed':SEED,
    'train_languages':list(TRAIN_LANGUAGES),'evaluation_languages':list(LANGUAGES),
    'original_case_ids':[r['source_case_id'] for r in cases],
    'warning':'questions machine translated: needs manual QA; not a new native multilingual benchmark'
},indent=2,ensure_ascii=False))
train=payload['train']
retain_reference=payload['retain_reference']
tokenizer=get_tokenizer()
max_len=max(make_qa(e,tokenizer)['input_ids'].shape[-1] for e in train)
print('SFT examples:',len(train),'retrain ref:',len(retain_reference),'max token length:',max_len)
assert all(r['source_case_id'] is not None for r in cases)
assert not any(x['relation']=='multi' and x['split']=='forget' for x in train)

In [ ]:
def train_adapter(rows,name,seed_shift=0):
    run = start_tracking("sft", name, {"seed": SEED+seed_shift, "examples": len(rows), "epochs": EPOCHS, "learning_rate": SFT_LR, "accumulation_steps": ACCUM, "lora_rank": LORA_R, "backbone": MODEL_ID, "dataset": "MQuAKE-T"})
    run.define_metric("train/optimizer_step")
    run.define_metric("train/*", step_metric="train/optimizer_step")
    optim_step = 0
    random.seed(SEED+seed_shift);torch.manual_seed(SEED+seed_shift)
    base=get_backbone()
    base.config.use_cache=False
    base=prepare_model_for_kbit_training(base,use_gradient_checkpointing=True)
    config=LoraConfig(r=LORA_R,lora_alpha=2*LORA_R,lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',
                      target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
    model=get_peft_model(base,config)
    model.train()
    optimizer=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),lr=SFT_LR)
    for epoch in range(EPOCHS):
        order=list(range(len(rows)));random.shuffle(order)
        optimizer.zero_grad(set_to_none=True)
        rolling=0
        for j,i in enumerate(order):
            b=to_gpu(make_qa(rows[i],tokenizer))
            loss=model(**b).loss
            (loss/ACCUM).backward()
            rolling+=float(loss.detach())
            if (j+1)%ACCUM==0 or j+1==len(order):
                torch.nn.utils.clip_grad_norm_(model.parameters(),1)
                optimizer.step();optimizer.zero_grad(set_to_none=True)
                optim_step += 1
                run.log({"train/optimizer_step": optim_step, "train/ce_loss": float(loss.detach()), "train/epoch_fraction": epoch+(j+1)/len(order), "train/lr": SFT_LR, "system/gpu_peak_gib": torch.cuda.max_memory_allocated(0)/1024**3})
            if (j+1)%100==0 or j+1==len(order):
                print(name,'epoch',epoch+1,'rows',j+1,'/',len(order),'mean_ce',round(rolling/(j+1),4))
    model.save_pretrained(ROOT/name,safe_serialization=True)
    tokenizer.save_pretrained(ROOT/name)
    print('Saved',ROOT/name)
    run.summary["train/final_ce_loss"] = float(loss.detach())
    run.summary["train/optimizer_steps"] = optim_step
    run.finish()
    return model

full=train_adapter(train,'full_sft_adapter')

In [ ]:
# Prerequisite evaluation on the SAME source IDs and direct/held-out 2-hop queries.
# Robust preliminary competence check on a larger random sample.
# The 512 held-out cases are not used for training or hyperparameter selection.
SFT_SANITY_PER_SPLIT=48
sanity_cases=([x for x in cases if x['split']=='forget'][:SFT_SANITY_PER_SPLIT]
              +[x for x in cases if x['split']=='retain'][:SFT_SANITY_PER_SPLIT])
rows=[]
for r in sanity_cases:
    for lang in LANGUAGES:
        for kind in ('direct','multi'):
            q=r[f'{"target" if kind=="direct" else "multi"}_question_{lang}']
            gold=r['target_answer'] if kind=='direct' else r['multi_answer']
            aliases=r['target_aliases'] if kind=='direct' else r['multi_aliases']
            pred=generate_answer(full,tokenizer,q)
            rows.append(dict(checkpoint='full_sft',case_id=r['source_case_id'],split=r['split'],language=lang,probe=kind,
                             question=q,gold=gold,prediction=pred,depth=r['depth'],correct=match_answer(pred,gold,aliases)))
sanity_df = pd.DataFrame(rows)
sanity_df.to_csv(ROOT/'sft_sanity.csv',index=False)
sanity_by_slice = sanity_df.groupby(['split','language','probe'],as_index=False).agg(accuracy=('correct','mean'), count=('correct','size'))
print(sanity_by_slice.to_string(index=False))
sanity_run = start_tracking("evaluation", "sft-sanity-six-languages", {"checkpoint": "full_sft", "num_probed_cases": len(sanity_cases), "source_sha256": source_sha})
sanity_metrics = {"eval/"+str(row.split)+"/"+str(row.language)+"/"+str(row.probe)+"_accuracy": float(row.accuracy) for row in sanity_by_slice.itertuples()}
sanity_metrics["eval/predictions"] = wandb.Table(dataframe=sanity_df)
sanity_metrics["eval/accuracy_by_slice"] = wandb.Table(dataframe=sanity_by_slice)
sanity_run.log(sanity_metrics)
sanity_run.finish()
print("Check EN/DE competence prerequisites in Notebook 02; RU/ZH/TR/VI are OUT-OF-TRAINING probes.")
print("WARNING: correct multi-hop FINAL answer does not prove exact first-hop recovery.")
print("Sanity includes",len(sanity_cases),"source facts and",len(sanity_df),"probe results.")
del full
cleanup()
if TRAIN_RETAIN_ONLY:
    retrain=train_adapter(retain_reference,'retain_only_adapter',seed_shift=1)
    del retrain
    cleanup()
print('Notebook 01 complete. Save Kaggle version with all outputs before Notebook 02.')

**Interpretation prerequisite:** Benchmark facts remain real temporal source facts, but data values can be outdated or ambiguous. Inspect `translation_audit.csv`, `sft_sanity.csv`, and representative benchmark source entries before proceeding. The retain-only adapter is an imperfect retraining reference because the pretrained backbone may already know the real-world target facts.